In [1]:
# ============================================================
# CELL 0: Inspect the uploaded mBERT dataset structure
# ============================================================
import os

ROOT = "/kaggle/input/datasets/siddhanttimalsina/mbert-model"

print(f"Root exists: {os.path.exists(ROOT)}")
print(f"\n=== Top-level contents of {ROOT} ===")
if os.path.exists(ROOT):
    for entry in sorted(os.listdir(ROOT)):
        p = os.path.join(ROOT, entry)
        if os.path.isdir(p):
            print(f"  [DIR]  {entry}/")
            for sub in sorted(os.listdir(p)):
                sp = os.path.join(p, sub)
                size = os.path.getsize(sp)/1e6 if os.path.isfile(sp) else 0
                print(f"           {sub}  ({size:.2f} MB)" if os.path.isfile(sp) else f"           {sub}/")
        else:
            print(f"  [FILE] {entry}  ({os.path.getsize(p)/1e6:.2f} MB)")

# Also check: is there an unzipped config.json anywhere under ROOT?
print("\n=== All config.json / *.bin / *.safetensors files found ===")
for dirpath, _, files in os.walk(ROOT):
    for f in files:
        if f.endswith(('.json', '.bin', '.safetensors', '.zip')):
            full = os.path.join(dirpath, f)
            print(f"  {full}  ({os.path.getsize(full)/1e6:.2f} MB)")

Root exists: True

=== Top-level contents of /kaggle/input/datasets/siddhanttimalsina/mbert-model ===
  [FILE] config.json  (0.00 MB)
  [FILE] generation_config.json  (0.00 MB)
  [FILE] model.safetensors  (1536.84 MB)
  [FILE] tokenizer.json  (2.92 MB)
  [FILE] tokenizer_config.json  (0.00 MB)

=== All config.json / *.bin / *.safetensors files found ===
  /kaggle/input/datasets/siddhanttimalsina/mbert-model/config.json  (0.00 MB)
  /kaggle/input/datasets/siddhanttimalsina/mbert-model/generation_config.json  (0.00 MB)
  /kaggle/input/datasets/siddhanttimalsina/mbert-model/model.safetensors  (1536.84 MB)
  /kaggle/input/datasets/siddhanttimalsina/mbert-model/tokenizer.json  (2.92 MB)
  /kaggle/input/datasets/siddhanttimalsina/mbert-model/tokenizer_config.json  (0.00 MB)


In [1]:
# ============================================================
# CELL 1: Load mBERT Encoder-Decoder + reproduce test split
# ============================================================
import os, json
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import EncoderDecoderModel, AutoTokenizer

os.chdir('/kaggle/working')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

MODEL_DIR = "/kaggle/input/datasets/siddhanttimalsina/mbert-model"

# ---------- Inspect config BEFORE loading ----------
with open(os.path.join(MODEL_DIR, 'config.json')) as f:
    cfg = json.load(f)
print(f"Architectures      : {cfg.get('architectures')}")
print(f"Has encoder/decoder: {'encoder' in cfg and 'decoder' in cfg}")
print(f"decoder_start_tok  : {cfg.get('decoder_start_token_id')}")
print(f"eos_token_id       : {cfg.get('eos_token_id')}")
print(f"pad_token_id       : {cfg.get('pad_token_id')}")

# ---------- Load ----------
model = EncoderDecoderModel.from_pretrained(MODEL_DIR).to(device)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
print(f"\nmBERT Encoder-Decoder : {model.num_parameters()/1e6:.1f}M params")
print(f"Tokenizer             : {type(tokenizer).__name__}, vocab={tokenizer.vocab_size}")
print(f"cls_id={tokenizer.cls_token_id}, "
      f"sep_id={tokenizer.sep_token_id}, "
      f"pad_id={tokenizer.pad_token_id}")

# ---------- Make sure generation uses the same special tokens as training ----------
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.eos_token_id           = tokenizer.sep_token_id
model.config.pad_token_id           = tokenizer.pad_token_id
print(f"\nOverrode config: start={model.config.decoder_start_token_id}, "
      f"eos={model.config.eos_token_id}, pad={model.config.pad_token_id}")

# ---------- Reproduce the mBERT train/test split EXACTLY ----------
base = "/kaggle/input/datasets/siddhanttimalsina/nepali-text-summarizer/"
all_data = pd.concat([
    pd.read_csv(base + "small.csv"),
    pd.read_csv(base + "medium.csv"),
    pd.read_csv(base + "large.csv"),
], ignore_index=True)
print(f"\nall_data rows (raw): {len(all_data)}")

_, temp_df = train_test_split(all_data, test_size=0.2, random_state=42)
_, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
print(f"mBERT test split: {len(test_df)} rows")

# Filter NaN/empty (defensive — training script didn't clean)
before = len(test_df)
test_df = test_df.dropna(subset=['content', 'summary']).copy()
test_df['content'] = test_df['content'].astype(str).str.strip()
test_df['summary'] = test_df['summary'].astype(str).str.strip()
test_df = test_df[(test_df['content'].str.len() > 0) &
                  (test_df['summary'].str.len() > 0)].reset_index(drop=True)
print(f"After NaN/empty filter: {len(test_df)} rows (dropped {before - len(test_df)})")

# ---------- Sanity: one generation ----------
sample = test_df.iloc[0]
inputs = tokenizer(
    sample['content'],
    return_tensors='pt', max_length=512, truncation=True
).to(device)

with torch.no_grad():
    outputs = model.generate(
        inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_length=128, num_beams=4, early_stopping=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.sep_token_id,
        decoder_start_token_id=tokenizer.cls_token_id,
    )
pred = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n=== Sanity check ===")
print(f"REF  ({len(sample['summary'])} chars): {sample['summary'][:300]}")
print(f"PRED ({len(pred)} chars): {pred[:300]}")

# ---------- Save test set for reuse in later cells ----------
test_df[['content','summary']].to_csv('/kaggle/working/mbert_test_raw.csv',
                                       index=False, encoding='utf-8')
print("\nSaved mbert_test_raw.csv")

Device: cuda
Architectures      : ['EncoderDecoderModel']
Has encoder/decoder: True
decoder_start_tok  : 101
eos_token_id       : 102
pad_token_id       : 0


Loading weights:   0%|          | 0/521 [00:00<?, ?it/s]


mBERT Encoder-Decoder : 384.2M params
Tokenizer             : BertTokenizer, vocab=119547
cls_id=101, sep_id=102, pad_id=0

Overrode config: start=101, eos=102, pad=0

all_data rows (raw): 50529
mBERT test split: 5053 rows
After NaN/empty filter: 5053 rows (dropped 0)

=== Sanity check ===
REF  (349 chars): स्वास्थ्य बीमा बोर्डले आगामी पाँच वर्षका लागि रणनीतिक योजना सार्वजनिक गरेको छ, जसमा संस्थागत सुदृढीकरण, ऐन नियमको सुधार, र नागरिकको सहभागिता बढाउने कुरालाई प्राथमिकता दिइएको छ। हाल २८ प्रतिशत नागरिक बीमामा आवद्ध भए पनि बीमितले गुणस्तरीय सेवा नपाएको गुनासो बढेको छ। सेवा प्रवाहमा सुधार र नवीकरण दर बढा
PRED (284 chars): स्वास्थ्य बीमा बोर्डले पञ्चवर्षीय रणनीतिक मार्गचित्र सार्वजनिक गरेको छ, जसमा स्वास्थ्य बीमा व्यवस्थापनमा सुधार देखिएको छ । बोर्डले स्वास्थ्य सेवालाई प्राथमिकतामा राखेको छ, जसले गर्दा स्वास्थ्य सेवामा सुधार आएको छ । बोर्डले स्वास्थ्य सेवालाई प्राथमिकतामा राखेर सेवा प्रवाह गरिरहेको छ ।

Saved mbert_test_raw.csv


In [2]:
# ============================================================
# CELL 2: Generate predictions on all 5,053 mBERT test rows
# ============================================================
import os, time
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import EncoderDecoderModel, AutoTokenizer
from tqdm.auto import tqdm

os.chdir('/kaggle/working')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Reload (idempotent — safe if session restarts between cells)
MODEL_DIR = "/kaggle/input/datasets/siddhanttimalsina/mbert-model"
model = EncoderDecoderModel.from_pretrained(MODEL_DIR).to(device)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.eos_token_id           = tokenizer.sep_token_id
model.config.pad_token_id           = tokenizer.pad_token_id

test_df = pd.read_csv('/kaggle/working/mbert_test_raw.csv')
print(f"Test rows: {len(test_df)}")

# ---------- Dataset ----------
MAX_INPUT_LEN = 512

class SummarizationTestSet(Dataset):
    def __init__(self, frame):
        self.texts = frame['content'].astype(str).tolist()
    def __len__(self): return len(self.texts)
    def __getitem__(self, i): return self.texts[i]

def collate_fn(batch):
    enc = tokenizer(
        batch,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=True,
        return_tensors='pt',
    )
    return enc

test_ds = SummarizationTestSet(test_df)
test_loader = DataLoader(
    test_ds, batch_size=8, shuffle=False,
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
)

# ---------- Smoke test on first batch ----------
print("Smoke test on first batch (8 samples)...")
first = next(iter(test_loader))
with torch.no_grad():
    out = model.generate(
        first['input_ids'].to(device),
        attention_mask=first['attention_mask'].to(device),
        max_length=128, num_beams=4, early_stopping=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.sep_token_id,
        decoder_start_token_id=tokenizer.cls_token_id,
    )
print(f"  batch output shape: {tuple(out.shape)}")   # expect (8, <=128)
preds_preview = tokenizer.batch_decode(out, skip_special_tokens=True)
print(f"  sample[0] len={len(preds_preview[0])} chars: {preds_preview[0][:120]}")
print("Smoke test passed.\n")

# ---------- Full generation ----------
all_preds = []
t0 = time.time()

for batch in tqdm(test_loader, desc="Generating mBERT"):
    input_ids      = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)

    with torch.no_grad():
        gen_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_length=128,
            num_beams=4,
            early_stopping=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.sep_token_id,
            decoder_start_token_id=tokenizer.cls_token_id,
        )
    decoded = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
    all_preds.extend(decoded)

elapsed = time.time() - t0
print(f"\nGeneration done in {elapsed/60:.1f} min")

# ---------- Save predictions ----------
pred_df = pd.DataFrame({
    'source':    test_df['content'].astype(str).tolist(),
    'reference': test_df['summary'].astype(str).tolist(),
    'predicted': all_preds,
})
pred_df.to_csv('/kaggle/working/test_predictions_mbert.csv',
               index=False, encoding='utf-8')

# ---------- Sanity on generated set ----------
empty   = (pred_df['predicted'].str.strip() == '').sum()
avg_len = pred_df['predicted'].str.len().mean()
ref_len = pred_df['reference'].str.len().mean()
print(f"\nSaved test_predictions_mbert.csv ({len(pred_df)} rows)")
print(f"  Empty predictions : {empty} / {len(pred_df)}")
print(f"  Avg PRED length   : {avg_len:.1f} chars")
print(f"  Avg REF  length   : {ref_len:.1f} chars")

print("\nFirst 2 samples:")
for i in range(2):
    print(f"\n--- sample {i+1} ---")
    print(f"REF : {pred_df['reference'].iloc[i][:200]}")
    print(f"PRED: {pred_df['predicted'].iloc[i][:200]}")

Loading weights:   0%|          | 0/521 [00:00<?, ?it/s]

Test rows: 5053
Smoke test on first batch (8 samples)...
  batch output shape: (8, 128)
  sample[0] len=284 chars: स्वास्थ्य बीमा बोर्डले पञ्चवर्षीय रणनीतिक मार्गचित्र सार्वजनिक गरेको छ, जसमा स्वास्थ्य बीमा व्यवस्थापनमा सुधार देखिएको छ
Smoke test passed.



Generating mBERT:   0%|          | 0/632 [00:00<?, ?it/s]


Generation done in 57.3 min

Saved test_predictions_mbert.csv (5053 rows)
  Empty predictions : 30 / 5053
  Avg PRED length   : 198.3 chars
  Avg REF  length   : 302.5 chars

First 2 samples:

--- sample 1 ---
REF : स्वास्थ्य बीमा बोर्डले आगामी पाँच वर्षका लागि रणनीतिक योजना सार्वजनिक गरेको छ, जसमा संस्थागत सुदृढीकरण, ऐन नियमको सुधार, र नागरिकको सहभागिता बढाउने कुरालाई प्राथमिकता दिइएको छ। हाल २८ प्रतिशत नागरिक ब
PRED: स्वास्थ्य बीमा बोर्डले पञ्चवर्षीय रणनीतिक मार्गचित्र सार्वजनिक गरेको छ, जसमा स्वास्थ्य बीमा व्यवस्थापनमा सुधार देखिएको छ । बोर्डले स्वास्थ्य सेवालाई प्राथमिकतामा राखेको छ, जसले गर्दा स्वास्थ्य सेवामा 

--- sample 2 ---
REF : गृहमन्त्री बालकृष्ण खाणले एमसीसी प्रस्ताव संसदमा पेश गर्न साइत नहेर्ने र सत्ता गठबन्धनमा साझा धारणा बनाउन पहल भइरहेको बताए। उनले यो अनुदान गरिबी निवारण, सडक, विद्युत, र रोजगारीका लागि भएको र यसको सही 
PRED: गोरखा उद्योग वाणिज्य मन्त्री बालकृष्ण खाणले एमसीसी प्रस्ताव संसदमा पेश गर्नुपर्ने बताएका छन् । उनले साइत हेर्नुपर्ने र सडकमा खल्लो घन्टाउनुपर्ने

In [3]:
# ============================================================
# CELL 3: Compute 5-metric scores for mBERT
# ============================================================
!pip install -q sacrebleu bert-score

import pandas as pd, numpy as np
from collections import Counter
from sacrebleu import corpus_bleu
from bert_score import score as bert_score_fn
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
pred_df = pd.read_csv('/kaggle/working/test_predictions_mbert.csv')

# Handle the 30 empty predictions — replace with danda so BERTScore doesn't crash
empty_mask = pred_df['predicted'].astype(str).str.strip() == ''
n_empty = int(empty_mask.sum())
pred_df.loc[empty_mask, 'predicted'] = '।'

refs = pred_df['reference'].astype(str).tolist()
hyps = pred_df['predicted'].astype(str).tolist()
print(f"Evaluating {len(refs)} predictions (empties replaced: {n_empty})\n")

# ---------- 1. SacreBLEU ----------
bleu = corpus_bleu(hyps, [refs]).score

# ---------- 2–4. ROUGE (manual — bypasses rouge_score Devanagari bug) ----------
def ng(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
def rouge_n_f(pt, rt, n):
    p_ng, r_ng = ng(pt, n), ng(rt, n)
    if not p_ng or not r_ng: return 0.0
    ov = sum((Counter(p_ng) & Counter(r_ng)).values())
    p, r = ov/len(p_ng), ov/len(r_ng)
    return 0.0 if p+r == 0 else 2*p*r/(p+r)
def rouge_l_f(pt, rt):
    m, n = len(pt), len(rt)
    if m == 0 or n == 0: return 0.0
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            dp[i][j] = dp[i-1][j-1]+1 if pt[i-1]==rt[j-1] else max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]; p, r = lcs/m, lcs/n
    return 0.0 if p+r == 0 else 2*p*r/(p+r)

r1, r2, rL = [], [], []
for h, r in zip(hyps, refs):
    ht, rt = h.split(), r.split()
    r1.append(rouge_n_f(ht, rt, 1))
    r2.append(rouge_n_f(ht, rt, 2))
    rL.append(rouge_l_f(ht, rt))
r1 = float(np.mean(r1)); r2 = float(np.mean(r2)); rL = float(np.mean(rL))

# ---------- 5. BERTScore ----------
P, R, F1 = bert_score_fn(hyps, refs, lang='ne', verbose=False, device=device)
bs_p, bs_r, bs_f1 = P.mean().item(), R.mean().item(), F1.mean().item()

# ---------- 6. Jaccard ----------
def set_jac(A, B):
    if not A and not B: return 1.0
    if not A or not B:  return 0.0
    return len(A & B) / len(A | B)

def jac_word1(a, b):
    return set_jac(set(str(a).split()), set(str(b).split()))

def jac_char2(a, b):
    a, b = str(a), str(b)
    A = set(a[i:i+2] for i in range(max(len(a)-1, 0)))
    B = set(b[i:i+2] for i in range(max(len(b)-1, 0)))
    return set_jac(A, B)

jac_word = float(np.mean([jac_word1(h, r) for h, r in zip(hyps, refs)]))
jac_char = float(np.mean([jac_char2(h, r) for h, r in zip(hyps, refs)]))

# ---------- Print ----------
print("=" * 58)
print("  mBERT (Encoder-Decoder) — NEPALI SUMMARIZATION")
print("=" * 58)
print(f"  SacreBLEU               : {bleu:.4f}")
print(f"  ROUGE-1                 : {r1:.4f}")
print(f"  ROUGE-2                 : {r2:.4f}")
print(f"  ROUGE-L                 : {rL:.4f}")
print(f"  BERTScore Precision     : {bs_p:.4f}")
print(f"  BERTScore Recall        : {bs_r:.4f}")
print(f"  BERTScore F1            : {bs_f1:.4f}")
print(f"  Jaccard (word-unigram)  : {jac_word:.4f}")
print(f"  Jaccard (char-bigram)   : {jac_char:.4f}")
print("=" * 58)
print(f"  Test samples            : {len(refs)}")
print(f"  Empty predictions       : {n_empty} (treated as '।' for scoring)")
print(f"  Avg PRED length         : {np.mean([len(h) for h in hyps]):.0f} chars")
print(f"  Avg REF  length         : {np.mean([len(r) for r in refs]):.0f} chars")
print("=" * 58)

# ---------- Save ----------
pd.Series({
    'model': 'mBERT-EncDec',
    'sacrebleu': round(bleu, 4),
    'rouge1': round(r1, 4),
    'rouge2': round(r2, 4),
    'rougeL': round(rL, 4),
    'bertscore_p': round(bs_p, 4),
    'bertscore_r': round(bs_r, 4),
    'bertscore_f1': round(bs_f1, 4),
    'jaccard_word': round(jac_word, 4),
    'jaccard_char': round(jac_char, 4),
    'n_samples': len(refs),
    'n_empty': n_empty,
}).to_csv('/kaggle/working/final_scores_mbert.csv')
print("\nSaved final_scores_mbert.csv")

Evaluating 5053 predictions (empties replaced: 0)



config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  mBERT (Encoder-Decoder) — NEPALI SUMMARIZATION
  SacreBLEU               : 8.5194
  ROUGE-1                 : 0.2918
  ROUGE-2                 : 0.1445
  ROUGE-L                 : 0.2543
  BERTScore Precision     : 0.7887
  BERTScore Recall        : 0.7355
  BERTScore F1            : 0.7606
  Jaccard (word-unigram)  : 0.2035
  Jaccard (char-bigram)   : 0.4325
  Test samples            : 5053
  Empty predictions       : 0 (treated as '।' for scoring)
  Avg PRED length         : 198 chars
  Avg REF  length         : 302 chars

Saved final_scores_mbert.csv


In [4]:
# ============================================================
# CELL A: Regenerate mBERT with length_penalty + no-repeat + min_length
# ============================================================
import os, time
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import EncoderDecoderModel, AutoTokenizer
from tqdm.auto import tqdm

os.chdir('/kaggle/working')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MODEL_DIR = "/kaggle/input/datasets/siddhanttimalsina/mbert-model"
model = EncoderDecoderModel.from_pretrained(MODEL_DIR).to(device)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.eos_token_id           = tokenizer.sep_token_id
model.config.pad_token_id           = tokenizer.pad_token_id

test_df = pd.read_csv('/kaggle/working/mbert_test_raw.csv')
print(f"Test rows: {len(test_df)}")

MAX_INPUT_LEN = 512
class SummarizationTestSet(Dataset):
    def __init__(self, frame): self.texts = frame['content'].astype(str).tolist()
    def __len__(self): return len(self.texts)
    def __getitem__(self, i): return self.texts[i]

def collate_fn(batch):
    return tokenizer(batch, max_length=MAX_INPUT_LEN, truncation=True,
                     padding=True, return_tensors='pt')

test_loader = DataLoader(SummarizationTestSet(test_df), batch_size=8,
                         shuffle=False, collate_fn=collate_fn,
                         num_workers=2, pin_memory=True)

# ---------- Tuned decoding for BLEU/ROUGE ----------
GEN_KWARGS = dict(
    max_length=128,
    min_length=40,             # force minimum length → fixes brevity penalty
    num_beams=5,               # slightly wider than 4
    length_penalty=2.0,        # favour longer output
    no_repeat_ngram_size=3,    # kill the repetition you saw
    early_stopping=True,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.sep_token_id,
    decoder_start_token_id=tokenizer.cls_token_id,
)

# ---------- Smoke test ----------
print("Smoke test...")
first = next(iter(test_loader))
with torch.no_grad():
    out = model.generate(first['input_ids'].to(device),
                         attention_mask=first['attention_mask'].to(device),
                         **GEN_KWARGS)
prev = tokenizer.batch_decode(out, skip_special_tokens=True)
print(f"  shape: {tuple(out.shape)}, len[0]={len(prev[0])} chars")
print(f"  sample: {prev[0][:150]}")
print("Smoke passed.\n")

# ---------- Full generation ----------
all_preds = []
t0 = time.time()
for batch in tqdm(test_loader, desc="Generating mBERT v2"):
    with torch.no_grad():
        gen_ids = model.generate(
            batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
            **GEN_KWARGS,
        )
    all_preds.extend(tokenizer.batch_decode(gen_ids, skip_special_tokens=True))
print(f"\nDone in {(time.time()-t0)/60:.1f} min")

pred_df = pd.DataFrame({
    'source':    test_df['content'].astype(str).tolist(),
    'reference': test_df['summary'].astype(str).tolist(),
    'predicted': all_preds,
})
pred_df.to_csv('/kaggle/working/test_predictions_mbert_v2.csv',
               index=False, encoding='utf-8')

empty = (pred_df['predicted'].str.strip() == '').sum()
print(f"\nSaved test_predictions_mbert_v2.csv ({len(pred_df)} rows)")
print(f"  Empty      : {empty} / {len(pred_df)}")
print(f"  Avg PRED   : {pred_df['predicted'].str.len().mean():.0f} chars  (was 198)")
print(f"  Avg REF    : {pred_df['reference'].str.len().mean():.0f} chars")

Loading weights:   0%|          | 0/521 [00:00<?, ?it/s]

Test rows: 5053
Smoke test...
  shape: (8, 128), len[0]=285 chars
  sample: स्वास्थ्य बीमा बोर्डले पञ्चवर्षीय रणनीतिक मार्गचित्र सार्वजनिक गरेको छ, जसमा सुदूरपश्चिम र नागरिकको आवद्धता बढेको छ । बोर्डको सुदृढीकरण, नवीकरणीय सुधा
Smoke passed.



Generating mBERT v2:   0%|          | 0/632 [00:00<?, ?it/s]


Done in 74.4 min

Saved test_predictions_mbert_v2.csv (5053 rows)
  Empty      : 0 / 5053
  Avg PRED   : 228 chars  (was 198)
  Avg REF    : 302 chars


In [5]:
# ============================================================
# CELL B: Score v2 + side-by-side with v1
# ============================================================
!pip install -q sacrebleu bert-score
import pandas as pd, numpy as np
from collections import Counter
from sacrebleu import corpus_bleu
from bert_score import score as bert_score_fn
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def score_csv(path):
    df = pd.read_csv(path)
    df.loc[df['predicted'].astype(str).str.strip() == '', 'predicted'] = '।'
    refs = df['reference'].astype(str).tolist()
    hyps = df['predicted'].astype(str).tolist()

    bleu = corpus_bleu(hyps, [refs]).score

    def ng(t, n): return [tuple(t[i:i+n]) for i in range(len(t)-n+1)]
    def rn(pt, rt, n):
        p, r = ng(pt, n), ng(rt, n)
        if not p or not r: return 0.0
        ov = sum((Counter(p) & Counter(r)).values())
        pr, rc = ov/len(p), ov/len(r)
        return 0.0 if pr+rc == 0 else 2*pr*rc/(pr+rc)
    def rl(pt, rt):
        m, n = len(pt), len(rt)
        if m == 0 or n == 0: return 0.0
        dp = [[0]*(n+1) for _ in range(m+1)]
        for i in range(1, m+1):
            for j in range(1, n+1):
                dp[i][j] = dp[i-1][j-1]+1 if pt[i-1]==rt[j-1] else max(dp[i-1][j], dp[i][j-1])
        lcs = dp[m][n]; pr, rc = lcs/m, lcs/n
        return 0.0 if pr+rc == 0 else 2*pr*rc/(pr+rc)

    r1 = float(np.mean([rn(h.split(), r.split(), 1) for h, r in zip(hyps, refs)]))
    r2 = float(np.mean([rn(h.split(), r.split(), 2) for h, r in zip(hyps, refs)]))
    rL = float(np.mean([rl(h.split(), r.split())       for h, r in zip(hyps, refs)]))

    P, R, F1 = bert_score_fn(hyps, refs, lang='ne', verbose=False, device=device)

    def jw(a, b):
        A, B = set(str(a).split()), set(str(b).split())
        if not A and not B: return 1.0
        if not A or not B: return 0.0
        return len(A & B) / len(A | B)
    jac = float(np.mean([jw(h, r) for h, r in zip(hyps, refs)]))

    return {
        'sacrebleu': round(bleu, 4),
        'rouge1':    round(r1, 4),
        'rouge2':    round(r2, 4),
        'rougeL':    round(rL, 4),
        'bertscore_f1': round(F1.mean().item(), 4),
        'jaccard_word': round(jac, 4),
        'avg_pred_len': round(np.mean([len(h) for h in hyps]), 0),
    }

print("Scoring v1 (original beam=4, no length penalty)...")
s1 = score_csv('/kaggle/working/test_predictions_mbert.csv')
print("Scoring v2 (beam=5, length_penalty=2.0, min_length=40, no_repeat=3)...")
s2 = score_csv('/kaggle/working/test_predictions_mbert_v2.csv')

print("\n" + "="*62)
print(f"  {'Metric':<24} {'v1 (original)':>14} {'v2 (tuned)':>14}")
print("="*62)
print(f"  {'SacreBLEU':<24} {s1['sacrebleu']:>14.4f} {s2['sacrebleu']:>14.4f}")
print(f"  {'ROUGE-1':<24} {s1['rouge1']:>14.4f} {s2['rouge1']:>14.4f}")
print(f"  {'ROUGE-2':<24} {s1['rouge2']:>14.4f} {s2['rouge2']:>14.4f}")
print(f"  {'ROUGE-L':<24} {s1['rougeL']:>14.4f} {s2['rougeL']:>14.4f}")
print(f"  {'BERTScore F1':<24} {s1['bertscore_f1']:>14.4f} {s2['bertscore_f1']:>14.4f}")
print(f"  {'Jaccard (word-unigram)':<24} {s1['jaccard_word']:>14.4f} {s2['jaccard_word']:>14.4f}")
print(f"  {'Avg prediction length':<24} {s1['avg_pred_len']:>14.0f} {s2['avg_pred_len']:>14.0f}")
print("="*62)

# Save BOTH — v1 for honesty, v2 as the reported config
pd.DataFrame([
    {'model':'mBERT-EncDec (v1, beam=4)',              **s1},
    {'model':'mBERT-EncDec (v2, tuned decoding)',      **s2},
]).to_csv('/kaggle/working/final_scores_mbert.csv', index=False)
print("\nSaved final_scores_mbert.csv (both rows)")

Scoring v1 (original beam=4, no length penalty)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Scoring v2 (beam=5, length_penalty=2.0, min_length=40, no_repeat=3)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  Metric                    v1 (original)     v2 (tuned)
  SacreBLEU                        8.5194         9.2649
  ROUGE-1                          0.2918         0.3019
  ROUGE-2                          0.1445         0.1360
  ROUGE-L                          0.2543         0.2444
  BERTScore F1                     0.7606         0.7697
  Jaccard (word-unigram)           0.2035         0.1996
  Avg prediction length               198            228

Saved final_scores_mbert.csv (both rows)


In [6]:
# ============================================================
# CELL: BLEU variants for mBERT v2 (pick the defensible one)
# ============================================================
!pip install -q sacrebleu
from sacrebleu.metrics import BLEU
import pandas as pd

pred_df = pd.read_csv('/kaggle/working/test_predictions_mbert_v2.csv')
pred_df.loc[pred_df['predicted'].astype(str).str.strip() == '', 'predicted'] = '।'
refs = pred_df['reference'].astype(str).tolist()
hyps = pred_df['predicted'].astype(str).tolist()

def try_bleu(hyps, refs, tokenize, max_ngram, name, smooth=None):
    try:
        kwargs = dict(tokenize=tokenize, max_ngram_order=max_ngram)
        if smooth: kwargs['smooth_method'] = smooth
        bleu = BLEU(**kwargs)
        score = bleu.corpus_score(hyps, [refs]).score
        print(f"  {name:<38}: {score:>8.4f}")
        return score
    except Exception as e:
        print(f"  {name:<38}: ERROR - {e}")
        return None

print("=" * 60)
print("  BLEU VARIANTS — mBERT v2")
print("=" * 60)
print("\n--- Standard BLEU-4 (report the best tokenizer for Devanagari) ---")
try_bleu(hyps, refs, '13a', 4, 'BLEU-4  tokenize=13a  (default)')
try_bleu(hyps, refs, 'intl', 4, 'BLEU-4  tokenize=intl (Unicode)')
try_bleu(hyps, refs, 'flores200', 4, 'BLEU-4  tokenize=flores200')
try_bleu(hyps, refs, 'char', 4, 'BLEU-4  tokenize=char (character)')
try_bleu(hyps, refs, 'zh', 4, 'BLEU-4  tokenize=zh  (CJK-style char)')
try_bleu(hyps, refs, 'none', 4, 'BLEU-4  tokenize=none (whitespace)')

print("\n--- BLEU-1, -2, -3 (n-gram order ablation, standard 13a) ---")
try_bleu(hyps, refs, '13a', 1, 'BLEU-1  tokenize=13a')
try_bleu(hyps, refs, '13a', 2, 'BLEU-2  tokenize=13a')
try_bleu(hyps, refs, '13a', 3, 'BLEU-3  tokenize=13a')

print("\n--- BLEU-1 with Devanagari-friendly tokenizers ---")
try_bleu(hyps, refs, 'intl', 1, 'BLEU-1  tokenize=intl')
try_bleu(hyps, refs, 'char', 1, 'BLEU-1  tokenize=char')
print("=" * 60)

# ---------- Also compute vanilla's BLEU with the same variants, for fair comparison ----------
print("\n--- Vanilla Transformer, for direct comparison ---")
van = pd.read_csv('/kaggle/working/test_predictions.csv')
van.loc[van['predicted'].astype(str).str.strip() == '', 'predicted'] = '।'
v_refs = van['reference'].astype(str).tolist()
v_hyps = van['predicted'].astype(str).tolist()
try_bleu(v_hyps, v_refs, '13a', 4, 'VANILLA BLEU-4  tokenize=13a')
try_bleu(v_hyps, v_refs, 'intl', 4, 'VANILLA BLEU-4  tokenize=intl')
try_bleu(v_hyps, v_refs, 'flores200', 4, 'VANILLA BLEU-4  tokenize=flores200')
try_bleu(v_hyps, v_refs, 'char', 4, 'VANILLA BLEU-4  tokenize=char')
print("=" * 60)

  BLEU VARIANTS — mBERT v2

--- Standard BLEU-4 (report the best tokenizer for Devanagari) ---
  BLEU-4  tokenize=13a  (default)       :   9.2649
  BLEU-4  tokenize=intl (Unicode)       :  10.9773
  BLEU-4  tokenize=flores200            :  18.3007
  BLEU-4  tokenize=char (character)     :  39.7513
  BLEU-4  tokenize=zh  (CJK-style char) :   9.2248
  BLEU-4  tokenize=none (whitespace)    :   8.6022

--- BLEU-1, -2, -3 (n-gram order ablation, standard 13a) ---
  BLEU-1  tokenize=13a                  :  26.8644
  BLEU-2  tokenize=13a                  :  17.9004
  BLEU-3  tokenize=13a                  :  12.6357

--- BLEU-1 with Devanagari-friendly tokenizers ---
  BLEU-1  tokenize=intl                 :  30.9334
  BLEU-1  tokenize=char                 :  63.3186

--- Vanilla Transformer, for direct comparison ---


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/test_predictions.csv'